# Q1 — Land cover, two seasons

Sentinel-2 imagery, a 3 × 4 grid of 500 m cells, RGB chips, and a random forest trained on two seasons (Oct–Dec 2019 and Feb–Mar 2019).

* **Edit only cell 2.** Every other cell reads its settings from there.
* Run the cells in order. Cell 10 downloads `02_classification_stage1_outputs.zip`; cell 18 downloads `03_classification_outputs.zip`, the input to `04_comparison_and_sensitivity.ipynb`.
* Input file: `00_training_polygons_v3.geojson` (upload it to the Colab folder, or you are asked for it in cell 13).
* Visual labels are the analyst's own, made from the Oct–Dec chips only, before any model or comparator output.

In [ ]:
# 1. Install packages and import libraries
!pip -q install earthengine-api pyproj requests matplotlib rasterio scikit-learn==1.6.1 joblib

import os, io, csv, json, math, random, shutil, zipfile, platform, datetime

import numpy as np
import requests
import ee
import pyproj
from pyproj import Transformer
import rasterio
import matplotlib
import sklearn
import joblib
from PIL import Image, ImageDraw, ImageFont

print("earthengine-api", ee.__version__)
print("pyproj", pyproj.__version__, "| rasterio", rasterio.__version__)
print("scikit-learn", sklearn.__version__, "| numpy", np.__version__)
if sklearn.__version__ != "1.6.1":
    print("WARNING: scikit-learn is not 1.6.1. Restart the runtime (Runtime > Restart session) and run from cell 1 again.")

In [ ]:
# 2. Settings (the only cell to edit)

# --- Earth Engine and site
EE_PROJECT = "ee-sawargaonkaraniket"
SITE_NAME  = "nashik_site"

# --- Grid: south-west corner (longitude first), 4 columns x 3 rows of 500 m cells
SW_LON, SW_LAT = 73.60339, 20.069111
N_COLS, N_ROWS = 4, 3
CELL_SIZE_M    = 500          # 4 x 500 m = 2.0 km wide, 3 x 500 m = 1.5 km tall, 3 km2
PIXEL_M        = 10           # Sentinel-2 10 m grid; the corner is snapped to a multiple of this

# --- Imagery windows (end date excluded)
MAIN_START,   MAIN_END   = "2019-10-01", "2020-01-01"   # chips, visual labels and model inputs
SECOND_START, SECOND_END = "2019-02-01", "2019-04-01"   # model inputs only

S2_COLLECTION       = "COPERNICUS/S2_SR_HARMONIZED"
CS_COLLECTION       = "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"
MAX_SCENE_CLOUD_PCT = 35      # keep scenes with CLOUDY_PIXEL_PERCENTAGE <= 35 (whole tile)
CS_THRESHOLD        = 0.60    # keep pixels with cs_cdf >= 0.60; pixels with no clear observation stay masked
COMPOSITE_BANDS     = ["B4", "B3", "B2", "B8", "B11", "B12"]

# --- Chips (same fixed stretch for every chip; reflectance x 10000)
CHIP_MIN, CHIP_MAX = 0, 3000
CHIP_PX            = 512
OVERVIEW_PX        = 1024

# --- Reproducibility and output
RANDOM_SEED = 2019
OUT_DIR     = "outputs"

# --- Classes (CGLS-LC100 names and codes). CLASS_NAMES sets the order used in every output.
CLASSES = {
    20: "Shrubs",
    30: "Herbaceous vegetation",
    40: "Cropland",
    50: "Built-up",
    60: "Bare or sparse vegetation",
    80: "Permanent water",
}
CLASS_NAMES = ["Shrubs", "Herbaceous vegetation", "Cropland", "Built-up",
               "Bare or sparse vegetation", "Permanent water"]
EXCLUDED_CLASSES_NOTE = ("Forest (CGLS 111-126) and Herbaceous wetland (CGLS 90) are left out: "
                         "no reliable training patches.")
MIXED_THRESHOLD = 0.50   # one class only if its share is MORE than 50%, otherwise "Mixed (top and second)"

# --- My own visual labels (made from the Oct-Dec chips only, without AI)
VISUAL_LABELS = {
    "C01": {"main_label": "Permanent water",             "confidence": "High",
            "second_class": "", "second_share": "", "cue": "", "source": ""},
    "C02": {"main_label": "Shrubs",                      "confidence": "High",
            "second_class": "", "second_share": "", "cue": "", "source": ""},
    "C05": {"main_label": "Permanent water",             "confidence": "High",
            "second_class": "", "second_share": "", "cue": "", "source": ""},
    "C08": {"main_label": "Cropland",                    "confidence": "Medium",
            "second_class": "", "second_share": "", "cue": "", "source": ""},
    "C10": {"main_label": "Mixed (Built-up and Shrubs)", "confidence": "Medium",
            "second_class": "", "second_share": "", "cue": "", "source": ""},
    "C11": {"main_label": "Mixed (Cropland and Shrubs)", "confidence": "Medium",
            "second_class": "", "second_share": "", "cue": "", "source": ""},
}

# --- Training area and model
TRAIN_BUFFER_M  = 2000    # training area = grid plus this margin on every side
MIN_GAP_M       = 100     # warn if a training polygon is closer than this to the grid
POLYGONS_FILE   = "00_training_polygons_v3.geojson"   # used if present in the Colab folder, otherwise you are asked to upload
RF_TREES        = 300
CV_FOLDS        = 5
LOWCONF_PROB    = 0.50    # a pixel is low-confidence if its highest class probability is below this

In [ ]:
# 3. Earth Engine sign-in (interactive; no keys or tokens in the code)
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialised for project", EE_PROJECT)

In [ ]:
# 4. Grid: UTM zone, snapped corner, cells C01-C12, GeoJSON files
os.makedirs(OUT_DIR, exist_ok=True)

# UTM zone from the corner (gives EPSG:32643 for this site)
UTM_ZONE = int((SW_LON + 180) // 6) + 1
UTM_EPSG = (32600 if SW_LAT >= 0 else 32700) + UTM_ZONE
EPSG     = f"EPSG:{UTM_EPSG}"
to_utm   = Transformer.from_crs("EPSG:4326", EPSG, always_xy=True)
to_wgs   = Transformer.from_crs(EPSG, "EPSG:4326", always_xy=True)

# Snap the corner to the 10 m grid so cell edges follow Sentinel-2 pixel edges
SW_X_RAW, SW_Y_RAW = to_utm.transform(SW_LON, SW_LAT)
X0 = round(SW_X_RAW / PIXEL_M) * PIXEL_M
Y0 = round(SW_Y_RAW / PIXEL_M) * PIXEL_M
X1 = X0 + N_COLS * CELL_SIZE_M
Y1 = Y0 + N_ROWS * CELL_SIZE_M
GRID_BOUNDS = (X0, Y0, X1, Y1)
GRID_W = int((X1 - X0) / PIXEL_M)    # 200 px
GRID_H = int((Y1 - Y0) / PIXEL_M)    # 150 px

# Cells numbered left to right, north row first
CELLS = []
for r in range(N_ROWS):
    for c in range(N_COLS):
        idx  = r * N_COLS + c + 1
        xmin = X0 + c * CELL_SIZE_M
        ymax = Y1 - r * CELL_SIZE_M
        CELLS.append({
            "cell_id": f"C{idx:02d}", "row": r + 1, "col": c + 1,
            "xmin": xmin, "ymin": ymax - CELL_SIZE_M, "xmax": xmin + CELL_SIZE_M, "ymax": ymax,
            "utm_epsg": UTM_EPSG,
            # extra properties
            "idx": idx, "area_m2": CELL_SIZE_M * CELL_SIZE_M,
            "pixels": (CELL_SIZE_M // PIXEL_M) ** 2,
            "tif_row0": int((Y1 - ymax) / PIXEL_M),   # top-left pixel of the cell in the grid GeoTIFFs (cell 8)
            "tif_col0": int((xmin - X0) / PIXEL_M),
        })
CELL_IDS = [c["cell_id"] for c in CELLS]

def utm_box_to_wgs_ring(xmin, ymin, xmax, ymax):
    corners = [(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax), (xmin, ymin)]
    return [[round(v, 7) for v in to_wgs.transform(x, y)] for x, y in corners]

def write_geojson(path, features):
    with open(path, "w") as f:
        json.dump({"type": "FeatureCollection", "features": features}, f, indent=1)

write_geojson(os.path.join(OUT_DIR, "aoi_boundary.geojson"), [{
    "type": "Feature",
    "properties": {"name": SITE_NAME, "xmin": X0, "ymin": Y0, "xmax": X1, "ymax": Y1,
                   "utm_epsg": UTM_EPSG, "area_km2": (X1 - X0) * (Y1 - Y0) / 1e6},
    "geometry": {"type": "Polygon", "coordinates": [utm_box_to_wgs_ring(*GRID_BOUNDS)]},
}])
write_geojson(os.path.join(OUT_DIR, "grid_cells.geojson"), [{
    "type": "Feature",
    "properties": c,
    "geometry": {"type": "Polygon", "coordinates": [utm_box_to_wgs_ring(c["xmin"], c["ymin"], c["xmax"], c["ymax"])]},
} for c in CELLS])

print(f"UTM zone {UTM_ZONE} -> {EPSG}")
print(f"SW corner raw ({SW_X_RAW:.2f}, {SW_Y_RAW:.2f}) -> snapped ({X0}, {Y0})")
print(f"Grid x {X0}-{X1}, y {Y0}-{Y1}: {(X1-X0)/1000} x {(Y1-Y0)/1000} km = {(X1-X0)*(Y1-Y0)/1e6} km2, "
      f"{GRID_W} x {GRID_H} px")

In [ ]:
# 5. Sentinel-2 image lists for both windows (searched over the study grid only)
def ee_rect(bounds):
    return ee.Geometry.Rectangle(list(bounds), proj=EPSG, geodesic=False)

AOI_EE   = ee_rect(GRID_BOUNDS)
CELLS_FC = ee.FeatureCollection([
    ee.Feature(ee_rect((c["xmin"], c["ymin"], c["xmax"], c["ymax"])), {"cell_id": c["cell_id"]})
    for c in CELLS])

WINDOWS = {
    "octdec": {"start": MAIN_START,   "end": MAIN_END,   "csv": "s2_images_used.csv"},
    "febmar": {"start": SECOND_START, "end": SECOND_END, "csv": "s2_images_used_febmar.csv"},
}

def s2_collection(start, end):
    return (ee.ImageCollection(S2_COLLECTION)
            .filterBounds(AOI_EE)
            .filterDate(start, end)
            .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT)))

def image_table(col):
    d = ee.Dictionary({
        "id":    col.aggregate_array("system:index"),
        "time":  col.aggregate_array("system:time_start"),
        "cloud": col.aggregate_array("CLOUDY_PIXEL_PERCENTAGE"),
        "tile":  col.aggregate_array("MGRS_TILE"),
    }).getInfo()
    rows = []
    for i, t, cc, tile in zip(d["id"], d["time"], d["cloud"], d["tile"]):
        date = datetime.datetime.fromtimestamp(t / 1000, datetime.timezone.utc).strftime("%Y-%m-%d")
        rows.append({"image_id": i, "date": date, "scene_cloud_pct": round(cc, 2), "mgrs_tile": tile})
    return sorted(rows, key=lambda r: (r["date"], r["image_id"]))

S2_COLS     = {}   # grid-only collections, reused by cells 6 and 12
IMAGE_LISTS = {}
for name, w in WINDOWS.items():
    S2_COLS[name] = s2_collection(w["start"], w["end"])
    rows = image_table(S2_COLS[name])
    IMAGE_LISTS[name] = rows
    with open(os.path.join(OUT_DIR, w["csv"]), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image_id", "date", "scene_cloud_pct", "mgrs_tile"])
        writer.writeheader()
        writer.writerows(rows)
    tiles = sorted({r["mgrs_tile"] for r in rows})
    print(f"{name}: {len(rows)} images ({w['start']} to {w['end']}, end excluded), tiles {tiles}")
    if not rows:
        print(f"WARNING: no images for {name}; check the dates and cloud limit in cell 2.")

In [ ]:
# 6. Cloud-masked median composites (grid-only collections) and per-cell valid data
CS = ee.ImageCollection(CS_COLLECTION)

def masked_collection(col):
    col = col.linkCollection(CS, ["cs_cdf"])
    return col.map(lambda img: img.updateMask(img.select("cs_cdf").gte(CS_THRESHOLD))
                                  .select(COMPOSITE_BANDS))

COMPOSITES = {}
for name in WINDOWS:
    masked = masked_collection(S2_COLS[name])
    COMPOSITES[name] = {
        "median": masked.median().toFloat(),                                        # masked where no clear obs
        "n_obs":  masked.select("B4").count().unmask(0).rename("n_obs").toFloat(),  # clear observations per pixel
    }

def cell_valid_stats(n_obs):
    img = n_obs.gt(0).rename("valid").addBands(n_obs.rename("nobs"))
    reducer = (ee.Reducer.sum()
               .combine(ee.Reducer.count(), sharedInputs=True)
               .combine(ee.Reducer.mean(), sharedInputs=True))
    fc = img.reduceRegions(collection=CELLS_FC, reducer=reducer, scale=PIXEL_M, crs=EPSG)
    return {f["properties"]["cell_id"]: f["properties"] for f in fc.getInfo()["features"]}

STATS = {name: cell_valid_stats(COMPOSITES[name]["n_obs"]) for name in WINDOWS}

VALID_ROWS = []
for c in CELLS:
    o, fm = STATS["octdec"][c["cell_id"]], STATS["febmar"][c["cell_id"]]
    total = int(round(o["valid_count"]))
    valid = int(round(o["valid_sum"]))
    VALID_ROWS.append({
        "cell_id":               c["cell_id"],
        "pixels_in_cell":        total,
        "valid_pixels":          valid,
        "valid_fraction":        round(valid / total, 4) if total else 0.0,
        "mean_clear_obs":        round(o["nobs_mean"], 3),
        "valid_pixels_febmar":   int(round(fm["valid_sum"])),
        "mean_clear_obs_febmar": round(fm["nobs_mean"], 3),
    })
    if total != c["pixels"]:
        print(f"WARNING: {c['cell_id']} has {total} pixels, expected {c['pixels']}")

with open(os.path.join(OUT_DIR, "stage1_cell_valid_data.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(VALID_ROWS[0].keys()))
    writer.writeheader()
    writer.writerows(VALID_ROWS)

for r in VALID_ROWS:
    print(f"{r['cell_id']}  Oct-Dec valid {r['valid_fraction']:.0%}, obs {r['mean_clear_obs']:.1f}"
          f"  |  Feb-Mar valid px {r['valid_pixels_febmar']}, obs {r['mean_clear_obs_febmar']:.1f}")

In [ ]:
# 7. RGB chips (fixed stretch 0-3000) for both windows, plus the Oct-Dec overview
CHIP_DIRS = {"octdec": os.path.join(OUT_DIR, "chips"), "febmar": os.path.join(OUT_DIR, "chips_febmar")}
for d in CHIP_DIRS.values():
    os.makedirs(d, exist_ok=True)

def rgb_vis(img):
    return img.visualize(bands=["B4", "B3", "B2"], min=CHIP_MIN, max=CHIP_MAX)

def save_thumb(img, region, path, px):
    url = rgb_vis(img).getThumbURL({"region": region, "dimensions": px, "format": "png", "crs": EPSG})
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)

for name in WINDOWS:
    for c in CELLS:
        region = ee_rect((c["xmin"], c["ymin"], c["xmax"], c["ymax"]))
        save_thumb(COMPOSITES[name]["median"], region,
                   os.path.join(CHIP_DIRS[name], f"{c['cell_id']}.png"), CHIP_PX)
    print(f"{name}: saved {len(CELLS)} chips to {CHIP_DIRS[name]}")

OVERVIEW_PATH = os.path.join(OUT_DIR, "overview_rgb.png")
save_thumb(COMPOSITES["octdec"]["median"], AOI_EE, OVERVIEW_PATH, OVERVIEW_PX)
print("Saved", OVERVIEW_PATH, Image.open(OVERVIEW_PATH).size)

In [ ]:
# 8. Download both composites as 10 m GeoTIFFs covering the grid only (200 x 150 px)
TIF_PATHS = {"octdec": os.path.join(OUT_DIR, "s2_composite_2019.tif"),
             "febmar": os.path.join(OUT_DIR, "s2_composite_2019_febmar.tif")}

def download_geotiff(img, path, bounds):
    """Download img (masked pixels left masked) on the 10 m grid covering bounds = (xmin, ymin, xmax, ymax)."""
    w = int((bounds[2] - bounds[0]) / PIXEL_M)
    h = int((bounds[3] - bounds[1]) / PIXEL_M)
    url = img.select(COMPOSITE_BANDS).getDownloadURL({
        "crs": EPSG,
        "crs_transform": [PIXEL_M, 0, bounds[0], 0, -PIXEL_M, bounds[3]],
        "dimensions": f"{w}x{h}",
        "format": "GEO_TIFF",
        "filePerBand": False,
    })
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    content = r.content
    if content[:2] == b"PK":   # zipped response: take the single .tif inside
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            content = z.read([n for n in z.namelist() if n.endswith(".tif")][0])
    with open(path, "wb") as f:
        f.write(content)
    with rasterio.open(path, "r+") as dst:
        dst.descriptions = tuple(COMPOSITE_BANDS)
    with rasterio.open(path) as src:
        t = src.transform
        ok = (src.crs.to_epsg() == UTM_EPSG and src.width == w and src.height == h and src.count == 6
              and abs(t.a - PIXEL_M) < 1e-9 and abs(t.e + PIXEL_M) < 1e-9
              and abs(t.c - bounds[0]) < 1e-6 and abs(t.f - bounds[3]) < 1e-6)
        print(f"{os.path.basename(path)}: {src.width}x{src.height} px, {src.count} bands "
              f"{list(src.descriptions)}, EPSG:{src.crs.to_epsg()}, origin ({t.c}, {t.f}) -> "
              f"{'grid check OK' if ok else 'WARNING: grid does not match the expected extent'}")
    return ok

def read_composite(path):
    """Return (6, H, W) float32 array with NaN where there is no clear observation, and the valid mask."""
    with rasterio.open(path) as src:
        arr = src.read(masked=True).astype("float32").filled(np.nan)
        transform = src.transform
    # No clear observation: masked in the file, or written as 0 in every band (never real reflectance)
    valid = np.isfinite(arr).all(axis=0) & ~(arr == 0).all(axis=0)
    arr[:, ~valid] = np.nan
    return arr, valid, transform

for name, path in TIF_PATHS.items():
    download_geotiff(COMPOSITES[name]["median"], path, GRID_BOUNDS)

In [ ]:
# 9. Overview with the grid and cell IDs
ov = Image.open(OVERVIEW_PATH).convert("RGBA")
bg = Image.new("RGBA", ov.size, (0, 0, 0, 255))
ov = Image.alpha_composite(bg, ov).convert("RGB")
W, H = ov.size
sx = W / (X1 - X0)
sy = H / (Y1 - Y0)

draw = ImageDraw.Draw(ov)
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", max(14, W // 40))
except OSError:
    font = ImageFont.load_default()

YELLOW = (255, 230, 0)
for k in range(N_COLS + 1):
    x = min(round(k * CELL_SIZE_M * sx), W - 1)
    draw.line([(x, 0), (x, H - 1)], fill=YELLOW, width=2)
for k in range(N_ROWS + 1):
    y = min(round(k * CELL_SIZE_M * sy), H - 1)
    draw.line([(0, y), (W - 1, y)], fill=YELLOW, width=2)
for c in CELLS:
    px = round((c["xmin"] - X0) * sx) + 8
    py = round((Y1 - c["ymax"]) * sy) + 6
    draw.text((px, py), c["cell_id"], fill=YELLOW, font=font, stroke_width=2, stroke_fill=(0, 0, 0))

OVERVIEW_GRID_PATH = os.path.join(OUT_DIR, "overview_with_grid.png")
ov.save(OVERVIEW_GRID_PATH)
print("Saved", OVERVIEW_GRID_PATH, ov.size)
ov

In [ ]:
# 10. Run log, then zip and download the stage 1 outputs
def strip_tile(rows):
    return [{"image_id": r["image_id"], "date": r["date"], "scene_cloud_pct": r["scene_cloud_pct"],
             "mgrs_tile": r["mgrs_tile"]} for r in rows]

run_log = {
    "site": SITE_NAME,
    "sw_corner_lon_lat": [SW_LON, SW_LAT],
    "crs": EPSG,
    "aoi_utm": {"xmin": X0, "ymin": Y0, "xmax": X1, "ymax": Y1},
    "grid": {"cols": N_COLS, "rows": N_ROWS, "cell_size_m": CELL_SIZE_M},
    "date_window": [MAIN_START, MAIN_END],
    "max_scene_cloud_pct": MAX_SCENE_CLOUD_PCT,
    "cloud_score_plus": {"band": "cs_cdf", "clear_threshold": CS_THRESHOLD},
    "composite": "per-pixel median of Cloud Score+ masked images; pixels with no clear observation left masked",
    "bands": COMPOSITE_BANDS,
    "chip_stretch": [CHIP_MIN, CHIP_MAX],
    "random_seed": RANDOM_SEED,
    "collections": {"sentinel2": S2_COLLECTION, "cloud_score_plus": CS_COLLECTION},
    "images_used": strip_tile(IMAGE_LISTS["octdec"]),
    "season2_window": [SECOND_START, SECOND_END],
    "images_used_season2": strip_tile(IMAGE_LISTS["febmar"]),
    "method": ("Random forest (scikit-learn) on 19 features from two seasonal Sentinel-2 composites "
               "(Oct-Dec 2019 and Feb-Mar 2019), trained on hand-drawn polygons outside the grid; "
               "cell label = class with share > MIXED_THRESHOLD, otherwise Mixed (top and second)"),
    # --- extra keys
    "image_counts": {"octdec": len(IMAGE_LISTS["octdec"]), "febmar": len(IMAGE_LISTS["febmar"])},
    "settings": {
        "EE_PROJECT": EE_PROJECT, "PIXEL_M": PIXEL_M, "window_end": "excluded",
        "scene_filter": "CLOUDY_PIXEL_PERCENTAGE <= max_scene_cloud_pct",
        "pixel_mask": "keep cs_cdf >= clear_threshold",
        "image_search_area": "study grid only (filterBounds on the grid rectangle)",
        "chip_px": CHIP_PX, "overview_px": OVERVIEW_PX,
        "chip_method": "ee.Image.visualize(B4/B3/B2, min, max) + getThumbURL",
        "CLASS_NAMES": CLASS_NAMES, "EXCLUDED_CLASSES_NOTE": EXCLUDED_CLASSES_NOTE,
        "MIXED_THRESHOLD": MIXED_THRESHOLD, "TRAIN_BUFFER_M": TRAIN_BUFFER_M, "RF_TREES": RF_TREES,
    },
    "grid_detail": {"utm_zone": UTM_ZONE, "sw_corner_raw_utm": [round(SW_X_RAW, 3), round(SW_Y_RAW, 3)],
                    "sw_corner_snapped_utm": [X0, Y0], "size_px": [GRID_W, GRID_H], "cells": CELLS},
    "stage1_outputs": sorted(os.path.relpath(os.path.join(dp, f), OUT_DIR)
                             for dp, _, fs in os.walk(OUT_DIR) for f in fs) + ["run_log.json"],
    "versions": {"python": platform.python_version(), "earthengine-api": ee.__version__,
                 "pyproj": pyproj.__version__, "rasterio": rasterio.__version__,
                 "scikit-learn": sklearn.__version__, "numpy": np.__version__},
}
with open(os.path.join(OUT_DIR, "run_log.json"), "w") as f:
    json.dump(run_log, f, indent=1, default=str)

zip_path = shutil.make_archive("02_classification_stage1_outputs", "zip", OUT_DIR)
print("Wrote", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab: download", zip_path, "manually.")

In [ ]:
# 11. Six cells for visual labelling: stratified by grid row, two random cells per row
rng = random.Random(RANDOM_SEED)
SIX_CELLS = []
for r in range(1, N_ROWS + 1):
    row_ids = [c["cell_id"] for c in CELLS if c["row"] == r]
    SIX_CELLS += sorted(rng.sample(row_ids, 2))
SIX_RULE = "stratified by grid row, two random cells per row"

with open(os.path.join(OUT_DIR, "six_cells_for_labelling.json"), "w") as f:
    json.dump({"random_seed": RANDOM_SEED, "rule": SIX_RULE,
               "sampler": "Python random.Random(seed).sample, rows taken north to south",
               "cells": SIX_CELLS}, f, indent=1)

print("Six cells for labelling:", SIX_CELLS)
if set(SIX_CELLS) != set(VISUAL_LABELS):
    print("WARNING: VISUAL_LABELS cells do not match the seeded selection", sorted(VISUAL_LABELS))

In [ ]:
# 12. Training-area composites (grid plus TRAIN_BUFFER_M), downloaded from Earth Engine
TRAIN_BOUNDS = (X0 - TRAIN_BUFFER_M, Y0 - TRAIN_BUFFER_M, X1 + TRAIN_BUFFER_M, Y1 + TRAIN_BUFFER_M)
TRAIN_W = int((TRAIN_BOUNDS[2] - TRAIN_BOUNDS[0]) / PIXEL_M)
TRAIN_H = int((TRAIN_BOUNDS[3] - TRAIN_BOUNDS[1]) / PIXEL_M)
TRAIN_TIF_PATHS = {"octdec": os.path.join(OUT_DIR, "training_area_composite.tif"),
                   "febmar": os.path.join(OUT_DIR, "training_area_composite_febmar2019.tif")}

# Same grid-only image collections and Cloud Score+ masking as cells 5-6
for name, path in TRAIN_TIF_PATHS.items():
    download_geotiff(COMPOSITES[name]["median"], path, TRAIN_BOUNDS)

write_geojson(os.path.join(OUT_DIR, "training_area.geojson"), [{
    "type": "Feature",
    "properties": {"name": f"{SITE_NAME}_training_area",
                   "xmin": TRAIN_BOUNDS[0], "ymin": TRAIN_BOUNDS[1],
                   "xmax": TRAIN_BOUNDS[2], "ymax": TRAIN_BOUNDS[3],
                   "utm_epsg": UTM_EPSG, "buffer_m": TRAIN_BUFFER_M},
    "geometry": {"type": "Polygon", "coordinates": [utm_box_to_wgs_ring(*TRAIN_BOUNDS)]},
}])
print(f"Training area x {TRAIN_BOUNDS[0]}-{TRAIN_BOUNDS[2]}, y {TRAIN_BOUNDS[1]}-{TRAIN_BOUNDS[3]}: "
      f"{TRAIN_W} x {TRAIN_H} px")

In [ ]:
# 13. Training polygons: load POLYGONS_FILE (or upload it), save to outputs/, number 1, 2, 3 ...
POLYGONS_OUT = os.path.join(OUT_DIR, "training_polygons.geojson")

if os.path.exists(POLYGONS_FILE):
    src_path = POLYGONS_FILE
else:
    from google.colab import files
    print(f"{POLYGONS_FILE} not found: choose your polygon GeoJSON file.")
    uploaded = files.upload()
    src_path = [n for n in uploaded if n.lower().endswith((".geojson", ".json"))][0]
with open(src_path) as f:
    poly_fc = json.load(f)
shutil.copyfile(src_path, POLYGONS_OUT)

# Class name from the feature properties (accepts a name or a CGLS code)
NAME_LOOKUP = {n.lower(): n for n in CLASS_NAMES}
def feature_class(props):
    for key in ("class", "class_name", "classname", "label", "name", "lc_class"):
        v = props.get(key)
        if isinstance(v, str) and v.strip().lower() in NAME_LOOKUP:
            return NAME_LOOKUP[v.strip().lower()]
    for key in ("class_code", "code", "cgls_code", "class"):
        v = props.get(key)
        try:
            if int(v) in CLASSES:
                return CLASSES[int(v)]
        except (TypeError, ValueError):
            pass
    return None

# Reproject to UTM; the GeoJSON is assumed to be WGS84 unless it names a projected CRS
crs_name = (poly_fc.get("crs") or {}).get("properties", {}).get("name", "EPSG:4326")
is_projected = str(UTM_EPSG) in crs_name
def reproject_coords(coords):
    if isinstance(coords[0], (int, float)):
        return list(coords[:2]) if is_projected else list(to_utm.transform(coords[0], coords[1]))
    return [reproject_coords(c) for c in coords]

POLYGONS = []        # poly_id = 1, 2, 3 ... in file order; any id field in the file is ignored
for k, feat in enumerate(poly_fc["features"], start=1):
    cls = feature_class(feat.get("properties") or {})
    geom = feat["geometry"]
    POLYGONS.append({"poly_id": k, "class": cls,
                     "geometry": {"type": geom["type"], "coordinates": reproject_coords(geom["coordinates"])}})
    if cls is None:
        print(f"WARNING: polygon {k} has no recognised class (properties {feat.get('properties')}); it is skipped.")

drawn = {n: sum(p["class"] == n for p in POLYGONS) for n in CLASS_NAMES}
print(f"Loaded {len(POLYGONS)} polygons from {src_path} -> saved as {POLYGONS_OUT}")
for n in CLASS_NAMES:
    print(f"  {n}: {drawn[n]} polygons")

In [ ]:
# 14. Training samples: 19 features per pixel inside the polygons
from rasterio.features import rasterize
from shapely.geometry import shape, box

FEATURE_BASE = ["B4", "B3", "B2", "B8", "B11", "B12", "NDVI", "NDBI", "MNDWI"]
FEATURES = ([f + "_octdec" for f in FEATURE_BASE] + [f + "_febmar" for f in FEATURE_BASE]
            + ["NDVI_change"])                                                  # 19 features

def season_features(arr):
    B4, B3, B2, B8, B11, B12 = arr
    with np.errstate(divide="ignore", invalid="ignore"):
        ndvi  = (B8 - B4) / (B8 + B4)
        ndbi  = (B11 - B8) / (B11 + B8)
        mndwi = (B3 - B11) / (B3 + B11)
    return [B4, B3, B2, B8, B11, B12, ndvi, ndbi, mndwi]

def feature_stack(arr_octdec, arr_febmar):
    """(19, H, W) float32 features; NDVI_change = NDVI(Oct-Dec) - NDVI(Feb-Mar)."""
    a = season_features(arr_octdec.astype("float32"))
    b = season_features(arr_febmar.astype("float32"))
    return np.stack(a + b + [a[6] - b[6]]).astype("float32")

ta_oct, ta_valid_oct, ta_transform = read_composite(TRAIN_TIF_PATHS["octdec"])
ta_feb, ta_valid_feb, _            = read_composite(TRAIN_TIF_PATHS["febmar"])
ta_feat = feature_stack(ta_oct, ta_feb)

# Rasterize in file order: where polygons overlap, the later polygon wins
usable = [p for p in POLYGONS if p["class"] is not None]
poly_raster = rasterize(((p["geometry"], p["poly_id"]) for p in usable),
                        out_shape=(TRAIN_H, TRAIN_W), transform=ta_transform, fill=0, dtype="int32")

# Warn about polygons closer than MIN_GAP_M to the grid
grid_box = box(*GRID_BOUNDS)
too_close = [(p["poly_id"], round(shape(p["geometry"]).distance(grid_box), 1))
             for p in usable if shape(p["geometry"]).distance(grid_box) < MIN_GAP_M]
if too_close:
    print(f"WARNING: {len(too_close)} polygon(s) closer than {MIN_GAP_M} m to the grid (poly_id, distance m):", too_close)

# Drop pixels inside the study grid, and pixels without valid data in both seasons
r0, c0 = int((TRAIN_BOUNDS[3] - Y1) / PIXEL_M), int((X0 - TRAIN_BOUNDS[0]) / PIXEL_M)
inside_grid = np.zeros((TRAIN_H, TRAIN_W), bool)
inside_grid[r0:r0 + GRID_H, c0:c0 + GRID_W] = True
labelled = poly_raster > 0
PIXELS_DROPPED_INSIDE_GRID = int((labelled & inside_grid).sum())
valid_both = ta_valid_oct & ta_valid_feb & np.isfinite(ta_feat).all(axis=0)
PIXELS_DROPPED_NO_DATA = int((labelled & ~inside_grid & ~valid_both).sum())
keep = labelled & ~inside_grid & valid_both

rows_idx, cols_idx = np.nonzero(keep)                       # raster order, north to south
poly_class = {p["poly_id"]: p["class"] for p in usable}
SAMPLE_POLY  = poly_raster[rows_idx, cols_idx].astype(int)
SAMPLE_CLASS = [poly_class[k] for k in SAMPLE_POLY]
SAMPLE_X     = ta_feat[:, rows_idx, cols_idx].T             # float32, unrounded (rounded only in the CSV)

with open(os.path.join(OUT_DIR, "training_samples.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["polygon_id", "class"] + FEATURES)
    for pid, cls, x in zip(SAMPLE_POLY, SAMPLE_CLASS, SAMPLE_X):
        w.writerow([int(pid), cls] + [f"{float(v):.4f}" for v in x])

print(f"Training pixels kept: {len(SAMPLE_POLY)}  |  dropped inside grid: {PIXELS_DROPPED_INSIDE_GRID}"
      f"  |  dropped without valid data in both seasons: {PIXELS_DROPPED_NO_DATA}")
for n in CLASS_NAMES:
    print(f"  {n}: {SAMPLE_CLASS.count(n)} pixels from {len(set(SAMPLE_POLY[np.array(SAMPLE_CLASS) == n]))} polygons")

In [ ]:
# 14b. Patch check: pixels, median NDVI and median red per polygon; greenness-order check
ndvi_i, red_i = FEATURES.index("NDVI_octdec"), FEATURES.index("B4_octdec")
sample_cls = np.array(SAMPLE_CLASS)
PATCH_ROWS = []
for p in usable:
    m = SAMPLE_POLY == p["poly_id"]
    n = int(m.sum())
    PATCH_ROWS.append({
        "polygon_id": p["poly_id"], "class": p["class"], "training_pixels": n,
        "median_ndvi": round(float(np.median(SAMPLE_X[m, ndvi_i])), 3) if n else "",
        "median_red":  int(round(float(np.median(SAMPLE_X[m, red_i])))) if n else "",
    })
with open(os.path.join(OUT_DIR, "training_patch_check.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["polygon_id", "class", "training_pixels", "median_ndvi", "median_red"])
    writer.writeheader()
    writer.writerows(PATCH_ROWS)

small = [(r["polygon_id"], r["training_pixels"]) for r in PATCH_ROWS if r["training_pixels"] < 4]
if small:
    print(f"WARNING: {len(small)} patch(es) with fewer than 4 training pixels (polygon_id, pixels):", small)
thin = [(n, int((sample_cls == n).sum())) for n in CLASS_NAMES if (sample_cls == n).sum() < 100]
if thin:
    print("WARNING: class(es) with fewer than 100 training pixels (class, pixels):", thin)

class_ndvi = {n: float(np.median(SAMPLE_X[sample_cls == n, ndvi_i])) if (sample_cls == n).any() else float("nan")
              for n in CLASS_NAMES}
print("Median Oct-Dec NDVI by class:", {n: round(v, 3) for n, v in class_ndvi.items()})
if class_ndvi["Cropland"] < class_ndvi["Shrubs"]:
    print("WARNING: Cropland's median NDVI is below Shrubs' - check the Cropland and Shrubs polygons.")
if class_ndvi["Shrubs"] < class_ndvi["Bare or sparse vegetation"]:
    print("WARNING: Shrubs' median NDVI is below Bare or sparse vegetation's - check those polygons.")

In [ ]:
# 15. Random forest: fit, out-of-bag score, cross-validation grouped by polygon
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import confusion_matrix

X = SAMPLE_X
y = np.array([CLASS_NAMES.index(c) for c in SAMPLE_CLASS])   # 0-5 in CLASS_NAMES order
groups = SAMPLE_POLY                                          # polygon numbers from cell 13

RF_PARAMS = {"n_estimators": RF_TREES, "random_state": RANDOM_SEED, "class_weight": "balanced", "n_jobs": -1}
rf = RandomForestClassifier(**RF_PARAMS, oob_score=True).fit(X, y)
OOB = round(float(rf.oob_score_), 4)
joblib.dump(rf, os.path.join(OUT_DIR, "rf_model.joblib"))

# Cross-validation that holds out whole polygons
cv_info = {"done": False, "folds": CV_FOLDS, "pixels": int(len(y)), "correct": None, "agreement": None}
try:
    cv = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    y_cv = cross_val_predict(RandomForestClassifier(**RF_PARAMS), X, y, groups=groups, cv=cv)
    correct = int((y_cv == y).sum())
    cv_info.update({"done": True, "correct": correct, "agreement": round(correct / len(y), 4)})
    cm = confusion_matrix(y, y_cv, labels=list(range(len(CLASS_NAMES))))
    with open(os.path.join(OUT_DIR, "rf_crossval_confusion.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["drawn_class \\ predicted_class"] + CLASS_NAMES + ["pixels"])
        for i, n in enumerate(CLASS_NAMES):
            w.writerow([n] + [int(v) for v in cm[i]] + [int(cm[i].sum())])
except ValueError as e:
    cv_info["reason_not_done"] = str(e)
    print("WARNING: grouped cross-validation not done:", e)

order = np.argsort(-rf.feature_importances_, kind="stable")
with open(os.path.join(OUT_DIR, "rf_feature_importance.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["feature", "importance"])
    for i in order:
        w.writerow([FEATURES[i], f"{rf.feature_importances_[i]:.4f}"])

training_summary = []
for n in CLASS_NAMES:
    m = sample_cls == n
    training_summary.append({"class": n, "polygons_drawn": drawn[n],
                             "polygons_used": int(len(set(SAMPLE_POLY[m]))),
                             "training_pixels": int(m.sum())})

RF_SETTINGS = {
    "model": "RandomForestClassifier",
    "sklearn_version": sklearn.__version__,
    "params": RF_PARAMS,
    "features": FEATURES,
    "classes": CLASS_NAMES,
    "training_summary": training_summary,
    "pixels_dropped_inside_grid": PIXELS_DROPPED_INSIDE_GRID,
    "oob_score": OOB,
    "cross_validation_by_patch": cv_info,
}
with open(os.path.join(OUT_DIR, "rf_settings.json"), "w") as f:
    json.dump(RF_SETTINGS, f, indent=1)

print(f"Training pixels {len(y)}, OOB score {OOB}")
if cv_info["done"]:
    print(f"Grouped {CV_FOLDS}-fold check: {cv_info['correct']}/{cv_info['pixels']} = {cv_info['agreement']:.4f} "
          f"(agreement with the drawn polygons, not accuracy)")

In [ ]:
# 16. Classify the grid (grid-only GeoTIFFs from cell 8) and summarise each cell
g_oct, g_valid_oct, g_transform = read_composite(TIF_PATHS["octdec"])
g_feb, g_valid_feb, _           = read_composite(TIF_PATHS["febmar"])
g_feat  = feature_stack(g_oct, g_feb)
g_valid = g_valid_oct & g_valid_feb & np.isfinite(g_feat).all(axis=0)

NCLS = len(CLASS_NAMES)
class_map = np.full((GRID_H, GRID_W), 255, dtype="uint8")
prob_map  = np.full((NCLS, GRID_H, GRID_W), np.nan, dtype="float32")
if g_valid.any():
    Xg = g_feat[:, g_valid].T                              # float32, unrounded, as in training
    proba = rf.predict_proba(Xg)                           # columns follow rf.classes_
    full = np.zeros((proba.shape[0], NCLS))
    full[:, rf.classes_] = proba
    class_map[g_valid] = full.argmax(axis=1).astype("uint8")
    prob_map[:, g_valid] = full.T.astype("float32")

profile = {"driver": "GTiff", "width": GRID_W, "height": GRID_H, "crs": EPSG, "transform": g_transform}
with rasterio.open(os.path.join(OUT_DIR, "rf_class_map.tif"), "w", count=1, dtype="uint8", nodata=255,
                   **profile) as dst:
    dst.write(class_map, 1)
    dst.update_tags(classes=json.dumps({i: n for i, n in enumerate(CLASS_NAMES)}))
with rasterio.open(os.path.join(OUT_DIR, "rf_class_probabilities.tif"), "w", count=NCLS, dtype="float32",
                   nodata=np.nan, **profile) as dst:
    dst.write(prob_map)
    dst.descriptions = tuple(CLASS_NAMES)

def mixed_label(top, second):
    return f"Mixed ({top} and {second})"

CELL_ROWS = []
for c in CELLS:
    r0, c0 = c["tif_row0"], c["tif_col0"]
    n = CELL_SIZE_M // PIXEL_M
    cm_, pm_, vm_ = class_map[r0:r0+n, c0:c0+n], prob_map[:, r0:r0+n, c0:c0+n], g_valid[r0:r0+n, c0:c0+n]
    nvalid = int(vm_.sum())
    v = VISUAL_LABELS.get(c["cell_id"], {})
    row = {"cell_id": c["cell_id"], "row": c["row"], "col": c["col"],
           "visual_label": v.get("main_label", ""), "visual_confidence": v.get("confidence", "")}
    if nvalid == 0:
        row.update({"rf_label": "No valid data", "rf_top_class": "", "rf_top_share": "",
                    "rf_second_class": "", "rf_second_share": "", "rf_mean_pixel_confidence": "",
                    "pixels_in_cell": n * n, "valid_pixels": 0})
        row.update({f"share_{k}": "" for k in CLASS_NAMES})
        row.update({f"meanprob_{k}": "" for k in CLASS_NAMES})
        lowconf = ""
    else:
        shares = np.array([(cm_[vm_] == i).sum() for i in range(NCLS)]) / nvalid
        order_ = np.argsort(-shares, kind="stable")
        top, sec = CLASS_NAMES[order_[0]], CLASS_NAMES[order_[1]]
        maxprob = pm_[:, vm_].max(axis=0)
        row.update({
            "rf_label": top if shares[order_[0]] > MIXED_THRESHOLD else mixed_label(top, sec),
            "rf_top_class": top, "rf_top_share": round(float(shares[order_[0]]), 4),
            "rf_second_class": sec, "rf_second_share": round(float(shares[order_[1]]), 4),
            "rf_mean_pixel_confidence": round(float(maxprob.mean()), 4),
            "pixels_in_cell": n * n, "valid_pixels": nvalid})
        row.update({f"share_{k}": round(float(shares[i]), 4) for i, k in enumerate(CLASS_NAMES)})
        row.update({f"meanprob_{k}": round(float(pm_[i][vm_].mean()), 4) for i, k in enumerate(CLASS_NAMES)})
        lowconf = round(float((maxprob < LOWCONF_PROB).mean()), 4)
    row.update({"rf_frac_lowconf": lowconf, "in_visual_sample": c["cell_id"] in SIX_CELLS,
                "visual_second_class": v.get("second_class", ""), "visual_second_share": v.get("second_share", ""),
                "visual_cue": v.get("cue", ""), "visual_source": v.get("source", "")})
    CELL_ROWS.append(row)

with open(os.path.join(OUT_DIR, "rf_cell_labels.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(CELL_ROWS[0].keys()))
    writer.writeheader()
    writer.writerows(CELL_ROWS)

for r in CELL_ROWS:
    share = f"{r['rf_top_share']:.1%}" if r["rf_top_share"] != "" else ""
    vis = f"   [visual: {r['visual_label']} / {r['visual_confidence']}]" if r["in_visual_sample"] else ""
    print(f"{r['cell_id']}  {r['rf_label']:<45} top share {share:>6}{vis}")

In [ ]:
# 17. Figure: Oct-Dec RGB (grid-only composite) beside the random-forest class map, with cell labels
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

CLASS_COLOURS = {"Shrubs": "#D55E00", "Herbaceous vegetation": "#F0E442", "Cropland": "#E69F00",
                 "Built-up": "#555555", "Bare or sparse vegetation": "#CC79A7", "Permanent water": "#0072B2"}  # Okabe-Ito

rgb = np.stack([g_oct[0], g_oct[1], g_oct[2]], axis=-1)            # B4, B3, B2
rgb = np.clip((rgb - CHIP_MIN) / (CHIP_MAX - CHIP_MIN), 0, 1)
rgb = np.nan_to_num(rgb, nan=0.0)
cls_plot = np.ma.masked_equal(class_map, 255)
extent = [X0, X1, Y0, Y1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
axes[0].imshow(rgb, extent=extent, interpolation="nearest")
axes[0].set_title(f"Sentinel-2 Oct-Dec 2019 median (B4/B3/B2, {CHIP_MIN}-{CHIP_MAX})")
axes[1].imshow(cls_plot, extent=extent, interpolation="nearest",
               cmap=ListedColormap([CLASS_COLOURS[n] for n in CLASS_NAMES]), vmin=-0.5, vmax=NCLS - 0.5)
axes[1].set_title("Random forest, two seasons (19 features)")

labels = {r["cell_id"]: r for r in CELL_ROWS}
for ax in axes:
    for x in range(X0, X1 + 1, CELL_SIZE_M):
        ax.axvline(x, color="white", lw=1)
    for yv in range(Y0, Y1 + 1, CELL_SIZE_M):
        ax.axhline(yv, color="white", lw=1)
    for c in CELLS:
        txt = c["cell_id"] + ("*" if c["cell_id"] in SIX_CELLS else "")
        if ax is axes[1]:
            txt += "\n" + labels[c["cell_id"]]["rf_label"].replace("Mixed (", "Mixed\n(")
        ax.text(c["xmin"] + 15, c["ymax"] - 15, txt, va="top", ha="left", fontsize=7.5, color="black",
                bbox=dict(facecolor="white", alpha=0.75, lw=0, pad=1.5))
    ax.set_xlim(X0, X1); ax.set_ylim(Y0, Y1)
    ax.set_xlabel(f"Easting (m, {EPSG})"); ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
axes[1].legend(handles=[Patch(color=CLASS_COLOURS[n], label=n) for n in CLASS_NAMES],
               loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
fig.text(0.01, 0.01, "* = cell in the visual-label sample", fontsize=8)
fig.tight_layout()
FIG_PATH = os.path.join(OUT_DIR, "rf_class_map.png")
fig.savefig(FIG_PATH, dpi=200, bbox_inches="tight")
print("Saved", FIG_PATH)
plt.show()

In [ ]:
# 18. Final run log and zip of all outputs
with open(os.path.join(OUT_DIR, "run_log.json")) as f:
    final_log = json.load(f)                     # keeps every top-level key from cell 10

final_log["stage2"] = {
    "six_cells": SIX_CELLS,
    "rule": SIX_RULE,
    "classes": CLASS_NAMES,
    "cgls_codes_per_class": {n: [code] for code, n in CLASSES.items()},
    "mixed_threshold": MIXED_THRESHOLD,
    "visual_labels": {k: {"label": v["main_label"], "confidence": v["confidence"]} for k, v in VISUAL_LABELS.items()},
    "visual_labels_detail": VISUAL_LABELS,
    "excluded_classes_note": EXCLUDED_CLASSES_NOTE,
}
final_log["stage3"] = {
    **RF_SETTINGS,
    "inputs": {"grid_composites": [os.path.basename(p) for p in TIF_PATHS.values()],
               "training_composites": [os.path.basename(p) for p in TRAIN_TIF_PATHS.values()],
               "training_polygons": os.path.basename(POLYGONS_OUT),
               "polygons_source_file": os.path.basename(src_path)},
    "training_buffer_m": TRAIN_BUFFER_M,
    "training_area_utm": {"xmin": TRAIN_BOUNDS[0], "ymin": TRAIN_BOUNDS[1],
                          "xmax": TRAIN_BOUNDS[2], "ymax": TRAIN_BOUNDS[3]},
    "aggregation": (f"Per cell, the share of valid pixels (valid in both seasons) in each predicted class; "
                    f"label = top class if its share > {MIXED_THRESHOLD}, otherwise 'Mixed (top and second)'; "
                    f"'No valid data' if the cell has no valid pixels. Low-confidence pixel: highest class "
                    f"probability < {LOWCONF_PROB}."),
    "pixels_dropped_no_valid_data": PIXELS_DROPPED_NO_DATA,
    "polygons_closer_than_min_gap": too_close,
}
final_log["all_outputs"] = sorted(os.path.relpath(os.path.join(dp, f), OUT_DIR)
                                  for dp, _, fs in os.walk(OUT_DIR) for f in fs)
with open(os.path.join(OUT_DIR, "run_log.json"), "w") as f:
    json.dump(final_log, f, indent=1, default=str)

zip_path = shutil.make_archive("03_classification_outputs", "zip", OUT_DIR)
print("Wrote", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB) with", len(final_log["all_outputs"]), "files")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab: download", zip_path, "manually.")